# Parte 1 · Noticias peruanas: recolección, limpieza y EDA

    **Medio elegido:** Opción A, La República. **Primera descarga comprobada:** 18 de octubre de 2026 (hora de Perú); la fecha y hora de cada nueva ejecución se registran en `datos/reporte_recoleccion.json` (UTC). **Fuentes base:** `https://larepublica.pe/rss/politica.xml`, `economia.xml`, `deportes.xml`, `ciencia.xml`; respaldo en sitemaps y páginas del medio. Como las fuentes oficiales del día solo dieron 53 noticias únicas, [Free News API](https://freenewsapi.ai/docs/search) sirve para descubrir URLs históricas. El recolector intenta verificar el HTML oficial y reporta cuándo solo obtuvo metadatos del índice. [Robots.txt](https://larepublica.pe/robots.txt) se consulta antes de cada artículo.

**Pregunta:** ¿Pueden convertirse los titulares peruanos en un clasificador de noticias? Este cuaderno produce el conjunto verificable y describe sus límites. Cada solicitud nueva espera 1,5 segundos y tiene un único reintento; los XML/HTML se guardan en `datos/raw/`.


In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display
from recolectar import recolectar, SECCIONES, COLUMNAS

RAIZ = Path.cwd()
reporte = recolectar(objetivo=400)
print(json.dumps(reporte, ensure_ascii=False, indent=2))
assert reporte['cumple_minimo_350'], 'Faltan noticias reales: revisar RSS, sitemaps e índice, sin rellenar datos.'


Índice: 1 páginas; 0 candidatos de las cuatro secciones.
Índice: 11 páginas; 242 candidatos de las cuatro secciones.
Índice: 21 páginas; 609 candidatos de las cuatro secciones.
Noticias únicas acumuladas: 75/400
Noticias únicas acumuladas: 100/400
Noticias únicas acumuladas: 125/400
Noticias únicas acumuladas: 150/400
Noticias únicas acumuladas: 175/400
Noticias únicas acumuladas: 200/400
Noticias únicas acumuladas: 225/400
Noticias únicas acumuladas: 250/400
Noticias únicas acumuladas: 275/400
Noticias únicas acumuladas: 300/400
Noticias únicas acumuladas: 325/400
Noticias únicas acumuladas: 350/400
Noticias únicas acumuladas: 375/400
Noticias únicas acumuladas: 400/400
{
  "descarga_utc": "2026-10-19T03:50:01.289946+00:00",
  "opcion": "A: La Republica",
  "fuentes": {
    "politica": {
      "url": "https://larepublica.pe/rss/politica.xml",
      "items": 23,
      "sitemap_anadidas": 0
    },
    "economia": {
      "url": "https://larepublica.pe/rss/economia.xml",
      "items": 6

## Limpieza y mapeo explícito

`recolectar.py` elimina titulares vacíos, URLs ajenas al medio, fechas ausentes, URLs y titulares repetidos. Reporta cada motivo; descarta cualquier sección fuera de las cuatro clases oficiales. La categoría es la sección editorial normalizada, por lo que puede haber casos ambiguos.


In [2]:
df = pd.read_csv(RAIZ / 'datos/noticias.csv', encoding='utf-8-sig')
mapa = pd.DataFrame([{'seccion_original': s, 'etiqueta': e} for s, e in
                      {**SECCIONES, 'tecnologia': 'ciencia_tecnologia'}.items()])
display(mapa)
display(pd.DataFrame(reporte['descartes'].items(), columns=['motivo', 'noticias_descartadas']))
assert list(df.columns) == COLUMNAS
assert len(df) >= 350 and df['url'].is_unique and df['titular'].notna().all()
assert df['etiqueta'].isin(set(SECCIONES.values())).all()
print('Forma:', df.shape)
display(df.head(5))


Forma: (399, 7)


,seccion_original,etiqueta
0,politica,politica
1,economia,economia
2,deportes,deportes
3,ciencia,ciencia_tecnologia
4,tecnologia,ciencia_tecnologia


,motivo,noticias_descartadas
0,enlace_sin_http_200_comprobado,1
1,indice_fuera_de_categorias,1879
2,indice_url_duplicada,42


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Comisión de Ética recomienda suspender por 120...,"Según el documento, la conducta atribuida al d...",La Republica,politica,https://larepublica.pe/politica/2026/10/18/com...,2026-10-18,politica
1,Keiko Fujimori coordinó con Delcy Rodríguez la...,La mandataria informó que su Gobierno iniciará...,La Republica,politica,https://larepublica.pe/politica/2026/10/18/kei...,2026-10-18,politica
2,Detención de seis policías por presunto robo r...,La situación ha llevado a que la desconfianza ...,La Republica,politica,https://larepublica.pe/politica/2026/10/18/det...,2026-10-18,politica
3,Seis regiones llegaron al 100% del conteo ofic...,"Amazonas, Ayacucho, Moquegua, Pasco, Puno y Ta...",La Republica,politica,https://larepublica.pe/politica/2026/10/18/6-r...,2026-10-18,politica
4,Comisión de Constitución del Senado continuará...,El debate se reanudará desde las 11 de la maña...,La Republica,politica,https://larepublica.pe/politica/2026/10/18/com...,2026-10-18,politica


## EDA · tres gráficos interactivos con conclusiones descriptivas generadas desde los datos


In [3]:
conteos = df['etiqueta'].value_counts().rename_axis('categoría').reset_index(name='noticias')
mayor = conteos.iloc[0]
fig = px.bar(conteos, x='categoría', y='noticias', color='categoría',
             title=f"{mayor['categoría']} reúne más noticias ({mayor['noticias']}) en la muestra")
fig.show()


Figure({
    'data': [{'hovertemplate': 'categoría=%{x}<br>noticias=%{y}<extra></extra>',
              'legendgroup': 'politica',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'politica',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['politica'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'bg==', 'dtype': 'i1'},
              'yaxis': 'y'},
             {'hovertemplate': 'categoría=%{x}<br>noticias=%{y}<extra></extra>',
              'legendgroup': 'deportes',
              'marker': {'color': '#EF553B', 'pattern': {'shape': ''}},
              'name': 'deportes',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['deportes'], dtype=object),
              'xaxis': 'x',
              'y': 

In [4]:
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
assert df['fecha'].notna().all(), 'Hay fechas sin analizar: revisar la fuente.'
por_mes = df.assign(mes=df['fecha'].dt.to_period('M').astype(str)).groupby('mes').size().reset_index(name='noticias')
pico = por_mes.loc[por_mes['noticias'].idxmax()]
fig = px.bar(por_mes, x='mes', y='noticias',
             title=f"{pico['mes']} concentra más titulares ({pico['noticias']}) en el período descargado")
fig.show()


Figure({
    'data': [{'hovertemplate': 'mes=%{x}<br>noticias=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['2026-09', '2026-10'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'JABrAQ==', 'dtype': 'i2'},
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': '2026-10 concentra más titulares (363) en el período descargado'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'mes'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'noticias'}}}
})

In [5]:
df['caracteres'] = df['titular'].str.len()
mediana = int(df['caracteres'].median())
fig = px.histogram(df, x='caracteres', nbins=30,
                   title=f"La longitud mediana del titular es {mediana} caracteres")
fig.show()


Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'caracteres=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'nbinsx': 30,
              'orientation': 'v',
              'showlegend': False,
              'type': 'histogram',
              'x': {'bdata': ('VABWAFMAPABbAHgAXwA9AEcASQBUAH' ... 'B9AGgAagB5AHEAdgB0AH0ASABlAGcA'),
                    'dtype': 'i2'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'La longitud mediana del titular es 103 caracteres'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'caracteres'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

## Comprobación de extremos y exportación


In [6]:
print('Cinco titulares más largos:')
display(df.nlargest(5, 'caracteres')[['titular', 'caracteres', 'url']])
print('Cinco titulares más cortos:')
display(df.nsmallest(5, 'caracteres')[['titular', 'caracteres', 'url']])
df[COLUMNAS].to_csv(RAIZ / 'datos/noticias.csv', index=False, encoding='utf-8-sig')
print('Archivo exportado:', df[COLUMNAS].shape)
display(df[COLUMNAS].head(5))


Cinco titulares más largos:
Cinco titulares más cortos:
Archivo exportado: (399, 7)


,titular,caracteres,url
44,"En 1986, la explosión del reactor de Chernóbil...",164,https://larepublica.pe/ciencia/2026/10/18/en-1...
36,"Pablo Guede ningunea a Franco Velazco, adminis...",149,https://larepublica.pe/deportes/2026/10/18/pab...
42,"Mia Shalit, arquera de Universitario, destaca ...",148,https://larepublica.pe/deportes/2026/10/16/mia...
136,Australia recuperó sus praderas marinas tras u...,148,https://larepublica.pe/ciencia/2026/10/12/aust...
43,Los lobos gigantes modificados genéticamente p...,147,https://larepublica.pe/ciencia/2026/10/18/los-...


,titular,caracteres,url
336,Minería ilegal elevó 8 veces la demanda de com...,54,https://larepublica.pe/politica/2026/10/10/min...
201,Vaticano informa que Papa León XIV tiene cánce...,56,https://larepublica.pe/politica/2026/10/13/vat...
247,Fixture de Alianza Lima en la Liga Peruana de ...,56,https://larepublica.pe/deportes/2026/10/09/fix...
135,¿Cuándo inicia la Liga Peruana de Vóley Femeni...,57,https://larepublica.pe/deportes/2026/10/13/cua...
313,Gobierno plantea regulación de armas de fuego ...,58,https://larepublica.pe/politica/2026/10/11/gob...


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Comisión de Ética recomienda suspender por 120...,"Según el documento, la conducta atribuida al d...",La Republica,politica,https://larepublica.pe/politica/2026/10/18/com...,2026-10-18,politica
1,Keiko Fujimori coordinó con Delcy Rodríguez la...,La mandataria informó que su Gobierno iniciará...,La Republica,politica,https://larepublica.pe/politica/2026/10/18/kei...,2026-10-18,politica
2,Detención de seis policías por presunto robo r...,La situación ha llevado a que la desconfianza ...,La Republica,politica,https://larepublica.pe/politica/2026/10/18/det...,2026-10-18,politica
3,Seis regiones llegaron al 100% del conteo ofic...,"Amazonas, Ayacucho, Moquegua, Pasco, Puno y Ta...",La Republica,politica,https://larepublica.pe/politica/2026/10/18/6-r...,2026-10-18,politica
4,Comisión de Constitución del Senado continuará...,El debate se reanudará desde las 11 de la maña...,La Republica,politica,https://larepublica.pe/politica/2026/10/18/com...,2026-10-18,politica


## Verificación de enlaces

El verificador reutiliza los HTML obtenidos con HTTP 200 y su fecha de descarga; solo consulta por red los enlaces sin esa evidencia, con pausas y caché de resultados. Una URL con dominio correcto no garantiza que el artículo siga publicado. Si hay fallos, revisar el archivo, corregir o excluir esas noticias y comprobar otra vez el mínimo de 350 filas.


In [7]:
from validar_urls import main as validar_enlaces
validar_enlaces()
comprobacion = pd.read_csv(RAIZ / 'datos/validacion_urls.csv', encoding='utf-8-sig')
display(comprobacion['resultado'].value_counts().rename_axis('resultado').reset_index(name='enlaces'))
display(comprobacion[comprobacion['resultado'] != 'ok'].head(20))
print('Comprobar y documentar los fallos antes de entregar; una respuesta 403 puede indicar bloqueo del medio.')


25/399 verificadas
50/399 verificadas
75/399 verificadas
100/399 verificadas
125/399 verificadas
150/399 verificadas
175/399 verificadas
200/399 verificadas
225/399 verificadas
250/399 verificadas
275/399 verificadas
300/399 verificadas
325/399 verificadas
350/399 verificadas
375/399 verificadas
Verificación terminada: 399 enlaces. Revisa C:\Users\victo\Documents\Codex\2026-10-18\referenced-chatgpt-conversation-this-is-an\outputs\tarea_1\datos\validacion_urls.csv para posibles fallos.
Comprobar y documentar los fallos antes de entregar; una respuesta 403 puede indicar bloqueo del medio.


,resultado,enlaces
0,ok,399


,url,codigo_http,resultado,fecha_utc


Verificar la fecha, ejecutar todas las celdas y guardar el notebook con salidas antes de subirlo a GitHub.
